# Fix: Mobility vs Aggression Scatter Plot

The original `scramble_rate_mobility` clustered everyone near zero because it only
counted scrambles on pass plays. This fix incorporates designed QB runs, rushing
yards per game, and rushing EPA to build a proper mobility score.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
# Run the mobility fix script first (only need to do this once)
!cd .. && PYTHONPATH=. python fix_mobility.py

Loading full play-by-play...

Mobility scores for 38 QBs:
                 passer_player_name  scramble_rate  designed_rush_rate  rush_yards_per_game  mobility_score
passer_player_id                                                                                           
00-0036945                 J.Fields            0.0            0.151852            16.555556        1.892963
00-0040691                   J.Dart            0.0            0.099034            16.833333        1.484738
00-0032268                M.Mariota            0.0            0.097378            14.000000        1.214470
00-0036389                  J.Hurts            0.0            0.100372            10.625000        1.058797
00-0034857                  J.Allen            0.0            0.084404            10.812500        0.986018
00-0034796                L.Jackson            0.0            0.076923            10.769231        0.890770
00-0039910                J.Daniels            0.0            0.072727        

In [3]:
# Load the improved mobility data and merge with existing features
features = pd.read_parquet('../data/processed/qb_features.parquet')
mobility = pd.read_parquet('../data/processed/qb_mobility.parquet')

# Merge the new mobility columns into features
new_cols = ['mobility_score', 'rush_yards_per_game', 'designed_rush_rate', 'total_mobility_rate']
for col in new_cols:
    if col in mobility.columns:
        features[col] = mobility[col]

features = features.fillna(0)
plot_df = features.reset_index()
plot_df.head()

,passer_player_id,player_name,team,pass_attempts,avg_air_yards,deep_ball_rate,avg_intended_epa,epa_clean,epa_pressure,completion_pct_pressure,...,scramble_rate_mobility,scramble_yards_per_attempt,comp_pct_short,comp_pct_medium,comp_pct_deep,overall_comp_pct,mobility_score,rush_yards_per_game,designed_rush_rate,total_mobility_rate
0,00-0023459,A.Rodgers,PIT,525,6.068548,0.091429,0.026640,0.228175,-0.350960,0.349398,...,0.0,0.0,0.738095,0.458333,0.326087,0.659274,-0.918044,0.000000,0.001901,0.001901
1,00-0026158,J.Flacco,CIN,430,7.398058,0.109302,-0.119176,0.177317,-0.549522,0.419643,...,0.0,0.0,0.691781,0.451219,0.315789,0.609223,-0.205887,1.833333,0.018265,0.018265
2,00-0026498,M.Stafford,LA,617,9.138047,0.145867,0.229150,0.408374,-0.027067,0.492063,...,0.0,0.0,0.732824,0.552846,0.410256,0.653199,-0.978779,0.000000,0.008039,0.008039
3,00-0029604,K.Cousins,ATL,279,7.026316,0.089606,-0.016175,0.099488,-0.000959,0.481481,...,0.0,0.0,0.728723,0.464286,0.136364,0.624060,-0.328608,0.900000,0.021053,0.021053
4,00-0030565,G.Smith,LV,501,6.195067,0.087824,-0.147133,0.256946,-0.537521,0.467890,...,0.0,0.0,0.758824,0.455882,0.342105,0.677130,-0.286122,0.400000,0.032819,0.032819


In [4]:
# FIXED: Mobility vs Aggression — now with real spread
fig = px.scatter(
    plot_df, 
    x='mobility_score', 
    y='avg_air_yards',
    text='player_name', 
    size='pass_attempts',
    color='clutch_epa',
    color_continuous_scale='RdYlGn',
    title='Mobility vs Aggression (colored by Clutch EPA)',
    labels={
        'mobility_score': 'Mobility Score (scrambles + designed runs + rush yards)',
        'avg_air_yards': 'Average Air Yards per Attempt',
        'clutch_epa': 'Clutch EPA'
    },
    hover_data=['team', 'rush_yards_per_game', 'designed_rush_rate']
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(
    width=1000, height=700,
    plot_bgcolor='white',
)

# Add quadrant lines at the median
fig.add_hline(y=plot_df['avg_air_yards'].median(), line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=plot_df['mobility_score'].median(), line_dash='dash', line_color='gray', opacity=0.5)

# Add quadrant labels
fig.add_annotation(x=plot_df['mobility_score'].max()*0.85, y=plot_df['avg_air_yards'].max()*0.98,
                   text='Mobile Gunslingers', showarrow=False, font=dict(size=11, color='gray'))
fig.add_annotation(x=plot_df['mobility_score'].min()*0.85, y=plot_df['avg_air_yards'].max()*0.98,
                   text='Pocket Gunslingers', showarrow=False, font=dict(size=11, color='gray'))
fig.add_annotation(x=plot_df['mobility_score'].max()*0.85, y=plot_df['avg_air_yards'].min()*1.02,
                   text='Mobile Game Managers', showarrow=False, font=dict(size=11, color='gray'))
fig.add_annotation(x=plot_df['mobility_score'].min()*0.85, y=plot_df['avg_air_yards'].min()*1.02,
                   text='Pocket Game Managers', showarrow=False, font=dict(size=11, color='gray'))

fig.show()

In [5]:
# Alternative view: Rush Yards/Game vs Air Yards — very intuitive
fig2 = px.scatter(
    plot_df,
    x='rush_yards_per_game',
    y='avg_air_yards',
    text='player_name',
    size='pass_attempts',
    color='avg_intended_epa',
    color_continuous_scale='RdYlGn',
    title='Rushing Production vs Downfield Aggression (colored by EPA)',
    labels={
        'rush_yards_per_game': 'Rush Yards per Game',
        'avg_air_yards': 'Average Air Yards per Attempt',
        'avg_intended_epa': 'EPA/Play'
    },
    hover_data=['team']
)
fig2.update_traces(textposition='top center', textfont_size=9)
fig2.update_layout(width=1000, height=700, plot_bgcolor='white')
fig2.show()

In [6]:
# Save updated features with new mobility columns
features.to_parquet('../data/processed/qb_features.parquet')
print('Updated features saved with new mobility metrics!')

Updated features saved with new mobility metrics!
